# Kustomize 01: bases and overlays

Kustomize has no variables and no templates: a base is plain YAML, an overlay references it and declares transformations. `kustomize create --autodetect` writes the first `kustomization.yaml` for you.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && rm -rf kustomize-lab && mkdir -p kustomize-lab/base kustomize-lab/overlays/dev kustomize-lab/overlays/prod && cd kustomize-lab/base
cat > deployment.yaml <<'YAML'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
spec:
  replicas: 1
  selector:
    matchLabels: {app: web}
  template:
    metadata:
      labels: {app: web}
    spec:
      containers:
        - name: web
          image: traefik/whoami:v1.11.0
          ports: [{name: http, containerPort: 80}]
YAML
cat > service.yaml <<'YAML'
apiVersion: v1
kind: Service
metadata:
  name: web
spec:
  selector: {app: web}
  ports: [{name: http, port: 80, targetPort: http}]
YAML
kustomize create --autodetect && cat kustomization.yaml && kustomize build . | grep -c '^kind:'


In [ ]:
cd /source/work/kustomize-lab/overlays/dev
kustomize create --resources ../../base --namespace web-dev --nameprefix dev- && kustomize edit set label environment:dev && cat kustomization.yaml && kustomize build . | yq '.metadata.name + " " + .metadata.namespace'


In [ ]:
cd /source/work/kustomize-lab/overlays/prod
kustomize create --resources ../../base --namespace web-prod --nameprefix prod- && kustomize edit set image traefik/whoami=traefik/whoami:v1.12.0 && kustomize edit set replicas web=3 && cat kustomization.yaml


In [ ]:
cd /source/work/kustomize-lab
kustomize build overlays/prod | yq 'select(.kind == "Deployment") | .spec.replicas, .spec.template.spec.containers[0].image'


`kustomize edit` commands write the same file you could edit by hand; that is the whole tool. The context is the overlay, and every environment-specific fact is spelled out once per overlay.
